The contents of this notebook were created with assistance from Claude generative AI.

# Tuned Classifiers — 5-fold Cross-Validation  (supersedes prior classifier notebooks)

Four families, 4-class stance `{anti, neutral, pro, off-topic}`:
**Naive Bayes · XGBoost · DistilRoBERTa@512 · ModernBERT@512** (both transformers at max_len **512**).

1. **Baseline** — library-default configs, trained once (load-if-saved).
2. **Hyperparameter tuning with 5-fold grouped cross-validation** — `StratifiedGroupKFold` on
   `link_id`; XGBoost/transformers via Optuna with a **5-fold-CV objective run trial-parallel across
   both GPUs**; Naive Bayes via `GridSearchCV` (cv = the 5 folds). Best config refit on the full pool
   and saved (load-if-saved), with hyperparameters persisted to `saved_models/tuned_hyperparams.json`.
3. **Gold** — tuned (and baseline) models on the held-out human gold set, population-weighted.

The heavy GPU work runs in `train_baseline.py` / `tune_5fold.py` (they own the multi-GPU
orchestration); these cells launch them and stream progress. **Kernel: `mads-m2-classifiers`.**

In [1]:
import os, sys
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
P15 = os.getcwd()
if P15 not in sys.path: sys.path.insert(0, P15)
import numpy as np, pandas as pd
import m5_common as M
F = M.load_frames(); df, gold = F["df"], F["gold_text"]
print(f"training pool = {len(df)}   gold = {len(gold)}   folds = {M.CV_N_SPLITS}")
print("label dist:", dict(df["label"].value_counts()))

training pool = 4830   gold = 301   folds = 5
label dist: {'pro': np.int64(1857), 'anti': np.int64(1292), 'neutral': np.int64(933), 'off-topic': np.int64(748)}


In [4]:
import os, sys, subprocess
from pathlib import Path
P15 = Path.cwd()
def launch(scriptargs, keep):
    cmd = [sys.executable, *[str(P15 / scriptargs[0])], *scriptargs[1:]]
    print("launching:", " ".join(cmd), "\n", flush=True)
    env = {**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false"}
    proc = subprocess.Popen(cmd, cwd=str(P15), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in proc.stdout:
        s = line.rstrip()
        if s.startswith(keep) or any(k in s for k in ("Error", "Traceback", "Exception", "!!")):
            print(s, flush=True)
    proc.wait(); print("\nexit code:", proc.returncode)

## Step 1 — Baseline models (library defaults)

Trains each family with out-of-the-box settings on the full training pool and saves to
`saved_models/baseline/`. Already-saved models are loaded (skipped). Set `RUN=False` to skip.

In [5]:
RUN = True
if RUN:
    launch(["train_baseline.py"], keep=("training pool", "baseline:", "NaiveBayes", "XGBoost",
                                        "DistilRoBERTa", "ModernBERT", "BASELINE DONE"))
else:
    print("RUN=False — skipping baseline training")

launching: <home>\.venvs\mads-m2-classifiers\Scripts\python.exe train_baseline.py 

training pool = 4830
NaiveBayes baseline: saved
XGBoost baseline: saved
DistilRoBERTa baseline: saved (39s)
ModernBERT baseline: saved (106s)
BASELINE DONE

exit code: 0


## Step 2 — Hyperparameter tuning with 5-fold CV (trial-parallel, both GPUs)

Optuna over the prior search spaces — **NB full grid, XGBoost 30 trials, each transformer 20 trials** —
with a **5-fold grouped-CV** objective. XGBoost/transformer trials run concurrently across GPU0 and
GPU1 (shared Optuna study); the best config is refit on the full pool and saved to
`saved_models/tuned/`. **This is the long step** (hours, dominated by the transformer 5-fold tuning).
Re-running skips families whose tuned model is already saved.

In [6]:
RUN = True
if RUN:
    launch(["tune_5fold.py", "--gpus", "0,1"],
           keep=("training pool", "precomputing", "== tuning", "== ", "launched GPU", "[", "trial",
                 "CV macro", "refit", "done", "skipping", "ALL TUNING DONE"))
else:
    print("RUN=False — skipping tuning")

launching: <home>\.venvs\mads-m2-classifiers\Scripts\python.exe tune_5fold.py --gpus 0,1 

training pool = 4830   models = ['NaiveBayes', 'XGBoost', 'DistilRoBERTa', 'ModernBERT']   folds = 5
precomputing XGBoost features (shared cache) ...
== NaiveBayes: tuned model already saved â€” skipping ==
== tuning XGBoost (5-fold CV) ==
[XGBoost] trial 2 state=COMPLETE value=0.4922 best=0.4949
[XGBoost] trial 3 state=COMPLETE value=0.4999 best=0.4999
[XGBoost] trial 4 state=COMPLETE value=0.4905 best=0.4999
[XGBoost] trial 6 state=COMPLETE value=0.5001 best=0.5001
[XGBoost] trial 7 state=COMPLETE value=0.4947 best=0.5001
[XGBoost] trial 5 state=COMPLETE value=0.5015 best=0.5015
[XGBoost] trial 9 state=COMPLETE value=0.5026 best=0.5026
[XGBoost] trial 8 state=COMPLETE value=0.5013 best=0.5026
[XGBoost] trial 11 state=COMPLETE value=0.4920 best=0.5026
[XGBoost] trial 10 state=COMPLETE value=0.5064 best=0.5064
[XGBoost] trial 13 state=COMPLETE value=0.5009 best=0.5064
[XGBoost] trial 14 state=COM

### 2b - more tuning

In [8]:
import os, sys, subprocess
from pathlib import Path
P15 = Path.cwd()
cmd = [sys.executable, str(P15/"tune_5fold.py"), "--model","ModernBERT","--add-trials","200","--gpus","0,1"]
env = {**os.environ, "PYTHONIOENCODING":"utf-8","PYTHONUNBUFFERED":"1","TOKENIZERS_PARALLELISM":"false"}
proc = subprocess.Popen(cmd, cwd=str(P15), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in proc.stdout:
    s = line.rstrip()
    if any(k in s for k in ("trial","==","best","refit","DONE","Error","Traceback","!!")): print(s, flush=True)
proc.wait(); print("exit", proc.returncode)

== ModernBERT: +200 trials (resuming from 20 existing) ==
  launched GPU0: 100 trials of ModernBERT
  launched GPU1: 100 trials of ModernBERT
[ModernBERT] trial 21 state=PRUNED value=0.5069 best=0.5551
[ModernBERT] trial 20 state=PRUNED value=0.5351 best=0.5551
[ModernBERT] trial 23 state=PRUNED value=0.5270 best=0.5551
[ModernBERT] trial 24 state=PRUNED value=0.5158 best=0.5551
[ModernBERT] trial 22 state=COMPLETE value=0.5539 best=0.5551
[ModernBERT] trial 25 state=PRUNED value=0.5200 best=0.5551
[ModernBERT] trial 27 state=PRUNED value=0.5015 best=0.5551
[ModernBERT] trial 26 state=COMPLETE value=0.5613 best=0.5613
[ModernBERT] trial 28 state=PRUNED value=0.5300 best=0.5613
[ModernBERT] trial 29 state=PRUNED value=0.5098 best=0.5613
[ModernBERT] trial 30 state=PRUNED value=0.5018 best=0.5613
[ModernBERT] trial 31 state=PRUNED value=0.5217 best=0.5613
[ModernBERT] trial 32 state=PRUNED value=0.5454 best=0.5613
[ModernBERT] trial 33 state=PRUNED value=0.5482 best=0.5613
[ModernBERT] t

## Step 3 — Tuned (and baseline) models on the gold set

Held-out human gold, **population-weighted**: macro-F1 (4-class), macro-F1 (3-class on-topic),
balanced accuracy, macro PR-AUC. The tuned table also shows the **5-fold CV macro-F1** (the tuning
objective) next to the gold numbers.

In [9]:
LABELS = M.LABELS
yg, wg = gold["label"], gold["w"].to_numpy()
hp = M.read_tuned_hp()
MET = ["macro_f1", "macro_f1_3class", "balanced_accuracy", "macro_p r_auc"]
PRETTY = {"macro_f1": "macroF1[4]", "macro_f1_3class": "macroF1[3]",
          "balanced_accuracy": "bal_acc", "macro_pr_auc": "macro_PR_AUC"}

rows = []
for stage in ["baseline", "tuned"]:
    for name in M.MODELS:
        if not M.exists(stage, name):
            continue
        proba = M.load_proba(stage, name, gold)
        pred  = np.asarray(LABELS)[proba.argmax(1)]
        mm = M.macro_metrics(yg, pred, sample_weight=wg)
        ap = M.ap_per_class(yg, proba, sample_weight=wg)
        row = {"stage": stage, "model": name, **{PRETTY[k]: round(mm.get(k, ap.get(k)), 3) for k in MET[:3]},
               "macro_PR_AUC": round(ap["macro_pr_auc"], 3)}
        if stage == "tuned":
            row["CV macroF1 (5-fold)"] = round(hp.get(name, {}).get("cv_macro_f1", float("nan")), 3)
        rows.append(row)
res = pd.DataFrame(rows)
print("Gold (population-weighted) — baseline vs 5-fold-CV-tuned:")
display(res.set_index(["stage", "model"]))

print("\nTUNED models — gold + the 5-fold CV objective they were selected on:")
display(res[res["stage"] == "tuned"].drop(columns="stage").set_index("model"))
res.to_csv("gold_results.csv", index=False)
print("\nsaved -> gold_results.csv")

Loading weights: 100%|██████████| 138/138 [00:00<00:00, 4181.91it/s]


Gold (population-weighted) — baseline vs 5-fold-CV-tuned:


macroF1[4]  macroF1[3]  bal_acc  macro_PR_AUC  \
stage    model                                                          
baseline NaiveBayes          0.180       0.248    0.319         0.356   
         XGBoost             0.299       0.377    0.426         0.462   
         DistilRoBERTa       0.357       0.475    0.458         0.471   
         ModernBERT          0.366       0.434    0.503         0.535   
tuned    NaiveBayes          0.315       0.368    0.418         0.379   
         XGBoost             0.352       0.418    0.463         0.495   
         DistilRoBERTa       0.361       0.454    0.443         0.469   
         ModernBERT          0.308       0.408    0.426         0.505   

                        CV macroF1 (5-fold)  
stage    model                               
baseline NaiveBayes                     NaN  
         XGBoost                        NaN  
         DistilRoBERTa                  NaN  
         ModernBERT                     NaN  
tuned    NaiveBayes                   0.430  
         XGBoost                      0.506  
         DistilRoBERTa                0.507  
         ModernBERT                   0.566


TUNED models — gold + the 5-fold CV objective they were selected on:


,macroF1[4],macroF1[3],bal_acc,macro_PR_AUC,CV macroF1 (5-fold)
model,,,,,
NaiveBayes,0.315,0.368,0.418,0.379,0.430
XGBoost,0.352,0.418,0.463,0.495,0.506
DistilRoBERTa,0.361,0.454,0.443,0.469,0.507
ModernBERT,0.308,0.408,0.426,0.505,0.566



saved -> gold_results.csv


bootstrap CI for gold-macro

In [2]:
# ── Bootstrap 95% CI on the gold macro-F1[4] gap: XGBoost − ModernBERT (tuned) ──
# Resamples the 301 gold items with replacement (carrying their population weights) and recomputes
# each model's population-weighted macro-F1[4] per resample → distribution of the gap. No retrain:
# predictions come from the saved tuned models. ~10-40s for N_BOOT=5000.
from sklearn.metrics import f1_score
rng = np.random.default_rng(M.SEED)
N_BOOT = 5000

yg = gold["label"].to_numpy()
wg = gold["w"].to_numpy()

# predicted labels from the saved tuned models (argmax of class-probabilities), computed once
preds = {name: np.asarray(M.LABELS)[M.load_proba("tuned", name, gold).argmax(1)]
         for name in ("XGBoost", "ModernBERT")}

def wmf1(idx, name):
    return f1_score(yg[idx], preds[name][idx], labels=M.LABELS, average="macro",
                    sample_weight=wg[idx], zero_division=0)

n = len(yg)
full = np.arange(n)
point = wmf1(full, "XGBoost") - wmf1(full, "ModernBERT")
diffs = np.array([(lambda i: wmf1(i, "XGBoost") - wmf1(i, "ModernBERT"))(rng.integers(0, n, n))
                  for _ in range(N_BOOT)])

lo, hi = np.percentile(diffs, [2.5, 97.5])
p_two = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
sig = "significant" if (lo > 0 or hi < 0) else "NOT significant"
print("Gold macro-F1[4] gap:  XGBoost − ModernBERT (tuned)")
print(f"  point estimate : {point:+.3f}")
print(f"  bootstrap mean : {diffs.mean():+.3f}")
print(f"  95% CI         : [{lo:+.3f}, {hi:+.3f}]   (N_BOOT={N_BOOT})")
print(f"  two-sided p    : {p_two:.3f}  →  {sig} at α=0.05")

<home>\.venvs\mads-m2-classifiers\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 138/138 [00:00<00:00, 5999.94it/s]


Gold macro-F1[4] gap:  XGBoost − ModernBERT (tuned)
  point estimate : +0.044
  bootstrap mean : +0.043
  95% CI         : [-0.011, +0.101]   (N_BOOT=5000)
  two-sided p    : 0.116  →  NOT significant at α=0.05


In [10]:
# ── CV mean ± std for each TUNED model (5-fold grouped CV) ────────────────────
# Refits each model's tuned hyperparameters on each of the 5 folds and scores all four metrics
# per fold -> mean ± std. Single-GPU; the two transformers refit 5x each, so expect ~20-30 min.
import os, sys, time
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1"); os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
P15 = os.getcwd()
if P15 not in sys.path: sys.path.insert(0, P15)
import numpy as np, pandas as pd, torch, joblib
from sklearn.base import clone
import m5_common as M

df = M.load_frames()["df"].reset_index(drop=True)
folds = M.fold_indices(df, M.CV_N_SPLITS)          # the same 5 grouped folds used for tuning
hp = M.read_tuned_hp()
LABELS, i2lab = M.LABELS, M.i2lab
MET = ["macro_f1", "macro_f1_3class", "balanced_accuracy", "macro_pr_auc"]

def fold_scores(model):
    rows = []
    if model == "XGBoost":
        X, y = M.get_xgb_cache(df)
        params = {k: v for k, v in hp["XGBoost"].items() if k != "cv_macro_f1"}
    elif model == "NaiveBayes":
        nb_base = joblib.load(M.model_path("tuned", "NaiveBayes"))   # carries the tuned hyperparameters
    else:
        name, mlen, bs = M.TF_SPECS[model]; thp = hp[model]
    for tr, va in folds:
        va_y = df.iloc[va]["label"]
        if model == "NaiveBayes":
            m = clone(nb_base); m.fit(df.iloc[tr][M.NB_COLS], df.iloc[tr]["label"])
            idx = [list(m.classes_).index(l) for l in LABELS]
            proba = m.predict_proba(df.iloc[va][M.NB_COLS])[:, idx]
            pred  = pd.Series(m.predict(df.iloc[va][M.NB_COLS]), index=df.iloc[va].index)
        elif model == "XGBoost":
            cwd = M.class_weights(pd.Series(y[tr])); sw = pd.Series(y[tr]).map(cwd).to_numpy()
            m = M.build_xgb(params, early_stop=True)
            m.fit(X[tr], y[tr], sample_weight=sw, eval_set=[(X[va], y[va])], verbose=False)
            m.get_booster().set_param({"device": "cpu"})
            proba = m.predict_proba(X[va]); pred = pd.Series(m.predict(X[va])).map(i2lab)
        else:  # transformer
            mdl, tok = M.train_tf(name, mlen, bs, df.iloc[tr], lr=thp["lr"], weight_decay=thp["weight_decay"],
                                  warmup_ratio=thp["warmup_ratio"], epochs=thp["epochs"], class_weighted=True)
            proba = M.predict_tf(mdl, tok, df.iloc[va], mlen, return_proba=True)
            pred  = pd.Series(np.asarray(LABELS)[proba.argmax(1)], index=df.iloc[va].index)
            del mdl; torch.cuda.empty_cache()
        mm = M.macro_metrics(va_y, pred); ap = M.ap_per_class(va_y, proba)
        rows.append({"macro_f1": mm["macro_f1"], "macro_f1_3class": mm["macro_f1_3class"],
                     "balanced_accuracy": mm["balanced_accuracy"], "macro_pr_auc": ap["macro_pr_auc"]})
    return pd.DataFrame(rows)

PRETTY = {"macro_f1": "CV macroF1[4]", "macro_f1_3class": "CV macroF1[3]",
          "balanced_accuracy": "CV bal_acc", "macro_pr_auc": "CV macro_PR_AUC"}
rows = {}
for model in M.MODELS:
    t0 = time.time(); fs = fold_scores(model)
    rows[model] = {PRETTY[k]: f"{fs[k].mean():.3f} ± {fs[k].std():.3f}" for k in MET}   # std = sample (ddof=1)
    print(f"{model}: done ({time.time()-t0:.0f}s)  CV macroF1[4] = {fs['macro_f1'].mean():.3f} ± {fs['macro_f1'].std():.3f}", flush=True)

cv_meanstd = pd.DataFrame(rows).T[[PRETTY[k] for k in MET]]
print("\n5-fold grouped CV — tuned config refit per fold (mean ± std):")
display(cv_meanstd)
cv_meanstd.to_csv("cv_meanstd.csv")
print("saved -> cv_meanstd.csv")

NaiveBayes: done (5s)  CV macroF1[4] = 0.430 ± 0.009
XGBoost: done (42s)  CV macroF1[4] = 0.506 ± 0.018


Loading weights: 100%|██████████| 101/101 [00:00<00:00, 9183.08it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|████

DistilRoBERTa: done (189s)  CV macroF1[4] = 0.501 ± 0.016


Loading weights: 100%|██████████| 136/136 [00:00<00:00, 8500.49it/s]
[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 136/136 [00:00<00:00, 10461.34it/s]
[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architect

ModernBERT: done (572s)  CV macroF1[4] = 0.542 ± 0.013

5-fold grouped CV — tuned config refit per fold (mean ± std):


,CV macroF1[4],CV macroF1[3],CV bal_acc,CV macro_PR_AUC
NaiveBayes,0.430 ± 0.009,0.517 ± 0.010,0.429 ± 0.009,0.456 ± 0.009
XGBoost,0.506 ± 0.018,0.599 ± 0.014,0.503 ± 0.018,0.536 ± 0.012
DistilRoBERTa,0.501 ± 0.016,0.563 ± 0.024,0.513 ± 0.015,0.542 ± 0.022
ModernBERT,0.542 ± 0.013,0.615 ± 0.023,0.538 ± 0.011,0.580 ± 0.014


saved -> cv_meanstd.csv
